# Gray-Box AIA72 — Supplementary 2026 Clean Future Evaluation

One-shot clean future evaluation of the frozen fallback-first v3 operational policy. **No tuning, reselection, or redesign is permitted in this notebook.**

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

# ============================================================
# Gray-Box AIA72 — Supplementary 2026 Clean Future Evaluation
# Frozen fallback-first v3 operational policy
# ============================================================
#
# This notebook performs the ONE-SHOT clean future evaluation on supplementary_2026.
#
# No tuning, threshold fitting, policy redesign, or policy reselection is allowed here.
# Primary policy is frozen as fallback_v3_q90.
# q80/q95 are sensitivity analyses only.

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

MOND_DIR = Path("/home/abmoses2000/graybox_aia72_conformal_mondrian_v2_20261005")
MOND_PROTOCOL = MOND_DIR / "protocol.json"
MOND_THRESHOLDS = MOND_DIR / "mondrian_thresholds.csv"

CUTS = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005/candidate_cutoffs.csv")

ALARM_DIR = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005")
ALARM_PROTOCOL = ALARM_DIR / "protocol.json"
ALARM_THRESHOLDS = ALARM_DIR / "alarm_thresholds.csv"

V3_DIR = Path("/home/abmoses2000/graybox_aia72_policy_v3_fallback_first_freeze_20261005")
V3_PROTOCOL = V3_DIR / "protocol.json"
V3_FAMILY = V3_DIR / "policy_family_v3.json"

OUT = Path("/home/abmoses2000/graybox_aia72_supplementary2026_v3_eval_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

ROLE = "supplementary_2026"
EXPECTED_ROWS = 11118
PRIMARY_ALPHA = 0.10
SEEDS = [17,29,43]
PCTS = [0.80,0.90,0.95]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond,msg):
    if not cond:
        raise ValueError(msg)

def entropy_binary(p):
    p=np.asarray(p,float)
    q=np.clip(p,1e-12,1-1e-12)
    return -(q*np.log(q)+(1-q)*np.log(1-q))

def mondrian_state(p,q0,q1):
    p=np.asarray(p,float)
    finite=np.isfinite(p)
    include0=finite & (p<=q0)
    include1=finite & ((1-p)<=q1)
    out=np.full(len(p),"unavailable",dtype=object)
    out[finite&(~include0)&(~include1)]="empty"
    out[include0&(~include1)]="singleton_no_flare"
    out[(~include0)&include1]="singleton_flare"
    out[include0&include1]="doubleton"
    return out,include0,include1

def binary_metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=int(((y==1)&(pred==1)).sum())
    fp=int(((y==0)&(pred==1)).sum())
    fn=int(((y==1)&(pred==0)).sum())
    tn=int(((y==0)&(pred==0)).sum())
    recall=tp/(tp+fn) if (tp+fn) else np.nan
    spec=tn/(tn+fp) if (tn+fp) else np.nan
    prec=tp/(tp+fp) if (tp+fp) else np.nan
    f1=2*prec*recall/(prec+recall) if np.isfinite(prec) and np.isfinite(recall) and (prec+recall)>0 else np.nan
    tss=recall+spec-1 if np.isfinite(recall) and np.isfinite(spec) else np.nan
    denom=((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
    hss=2*(tp*tn-fn*fp)/denom if denom else np.nan
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,recall=float(recall),specificity=float(spec),
                precision=float(prec),f1=float(f1),tss=float(tss),hss=float(hss))

def brier(y,p):
    m=np.isfinite(p)
    return float(np.mean((p[m]-y[m])**2)) if m.any() else None

def logloss(y,p):
    m=np.isfinite(p)
    if not m.any(): return None
    q=np.clip(p[m],1e-12,1-1e-12); yy=y[m]
    return float(-np.mean(yy*np.log(q)+(1-yy)*np.log(1-q)))

for p in [MASTER,SHARP,MOND_PROTOCOL,MOND_THRESHOLDS,CUTS,ALARM_PROTOCOL,ALARM_THRESHOLDS,V3_PROTOCOL,V3_FAMILY]:
    require(p.is_file(),f"Missing frozen input: {p}")

require(sha256(MASTER)==MASTER_SHA,"Master SHA changed")
require(sha256(SHARP)==SHARP_SHA,"SHARP SHA changed")

v3_protocol=json.loads(V3_PROTOCOL.read_text())
v3_family=json.loads(V3_FAMILY.read_text())
mond_protocol=json.loads(MOND_PROTOCOL.read_text())
alarm_protocol=json.loads(ALARM_PROTOCOL.read_text())

require(v3_protocol["status"]=="FALLBACK_FIRST_V3_OPERATIONAL_POLICY_FROZEN","v3 not frozen")
require(v3_protocol["primary_policy_id"]=="fallback_v3_q90","v3 primary changed")
require(v3_protocol["supplementary_2026_outcomes_read"] is False,"2026 already read during v3 freeze")
require(v3_protocol["supplementary_2026_role"]=="clean_future_evaluation","2026 role changed")
require(v3_family["supplementary_2026_used_for_design_or_tuning"] is False,"2026 contaminated")
require(mond_protocol["policy_validation_used_to_fit_v2"] is False,"Mondrian v2 contaminated")

cuts=pd.read_csv(CUTS)
mond=pd.read_csv(MOND_THRESHOLDS)
alarm=pd.read_csv(ALARM_THRESHOLDS)
alarm_map={r.branch:float(r.threshold) for r in alarm.itertuples()}

q={}
for branch in ["sharp","aia","sharp_aia"]:
    hit=mond.loc[(mond["branch"]==branch)&np.isclose(mond["alpha"].astype(float),PRIMARY_ALPHA)]
    require(len(hit)==1,f"Missing Mondrian threshold: {branch}")
    q[branch]=(float(hit.iloc[0]["qhat_class0"]),float(hit.iloc[0]["qhat_class1"]))

def cutoff(metric,pct):
    hit=cuts.loc[(cuts["metric"]==metric)&np.isclose(cuts["percentile"].astype(float),pct)]
    require(len(hit)==1,f"Missing cutoff {metric} q{pct}")
    return float(hit.iloc[0]["cutoff"])

# -------------------------
# Open supplementary 2026 ONCE
# -------------------------
master=pd.read_csv(MASTER,low_memory=False)
sharp=pd.read_csv(SHARP,low_memory=False)

a=master.loc[master["role"].astype(str)==ROLE].copy()
s=sharp.loc[sharp["role"].astype(str)==ROLE].copy()

require(len(a)==EXPECTED_ROWS and len(s)==EXPECTED_ROWS,
        f"2026 support changed: master={len(a)}, sharp={len(s)}")

a["forecast_case_id"]=a["forecast_case_id"].astype(str)
s["forecast_case_id"]=s["forecast_case_id"].astype(str)
require(a["forecast_case_id"].tolist()==s["forecast_case_id"].tolist(),"2026 ID/order mismatch")

y=pd.to_numeric(a["label"],errors="raise").astype(int).to_numpy()
ys=pd.to_numeric(s["label"],errors="raise").astype(int).to_numpy()
require(np.array_equal(y,ys),"2026 labels disagree")

aia_seed=a[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
sharp_seed=s[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
aia_p=pd.to_numeric(a["probability"],errors="coerce").to_numpy(float)
sharp_p=pd.to_numeric(s["probability_gru_mean"],errors="coerce").to_numpy(float)
fusion_seed=0.5*sharp_seed+0.5*aia_seed
fusion_p=0.5*sharp_p+0.5*aia_p

aia_status=a["input_status"].astype(str).to_numpy() if "input_status" in a.columns else np.array(["ok"]*len(a))
aia_ok=(aia_status=="ok")&np.isfinite(aia_p)&np.isfinite(aia_seed).all(axis=1)
sharp_ok=np.isfinite(sharp_p)&np.isfinite(sharp_seed).all(axis=1)
fusion_ok=aia_ok&sharp_ok&np.isfinite(fusion_p)

sig=pd.DataFrame({
    "forecast_case_id":a["forecast_case_id"].to_numpy(),
    "issue_utc":a["issue_utc"].to_numpy() if "issue_utc" in a.columns else pd.NA,
    "label":y,
    "aia_input_status":aia_status,
    "sharp_input_status":np.where(sharp_ok,"ok","unavailable"),
    "aia_probability":aia_p,
    "sharp_probability":sharp_p,
    "fusion_probability":fusion_p,
    "aia_seed_std":np.nanstd(aia_seed,axis=1,ddof=1),
    "sharp_seed_std":np.nanstd(sharp_seed,axis=1,ddof=1),
    "fusion_seed_std":np.nanstd(fusion_seed,axis=1,ddof=1),
    "sharp_aia_probability_gap":np.abs(sharp_p-aia_p),
    "fusion_entropy":entropy_binary(fusion_p),
})

# Mondrian diagnostic coverage on 2026.
coverage_rows=[]
for branch,p in [("sharp",sharp_p),("aia",aia_p),("sharp_aia",fusion_p)]:
    state,inc0,inc1=mondrian_state(p,*q[branch])
    sig[f"{branch}_mondrian_state"]=state
    contains=np.where(y==0,inc0,inc1)
    sig[f"{branch}_contains_true"]=contains
    finite=state!="unavailable"
    coverage_rows.append({
        "branch":branch,
        "alpha":PRIMARY_ALPHA,
        "available_cases":int(finite.sum()),
        "overall_coverage":float(contains[finite].mean()),
        "class0_coverage":float(contains[(y==0)&finite].mean()),
        "class1_coverage":float(contains[(y==1)&finite].mean()),
        "empty_rate":float((state[finite]=="empty").mean()),
        "singleton_no_flare_rate":float((state[finite]=="singleton_no_flare").mean()),
        "singleton_flare_rate":float((state[finite]=="singleton_flare").mean()),
        "doubleton_rate":float((state[finite]=="doubleton").mean()),
    })
coverage=pd.DataFrame(coverage_rows)

# Individual branch metrics.
branch_rows=[]
for branch,p,available in [
    ("sharp",sharp_p,sharp_ok),
    ("aia",aia_p,aia_ok),
    ("sharp_aia",fusion_p,fusion_ok)
]:
    pred=(p>=alarm_map[branch]).astype(int)
    m=binary_metrics(y[available],pred[available])
    branch_rows.append({
        "branch":branch,
        "available_cases":int(available.sum()),
        "availability_rate":float(available.mean()),
        "brier":brier(y,p),
        "log_loss":logloss(y,p),
        **m
    })
branch_metrics=pd.DataFrame(branch_rows)

# Baselines:
# 1) always issue fusion whenever fusion is available
# 2) always issue SHARP whenever SHARP is available
comparators=[]

fusion_pred=(fusion_p>=alarm_map["sharp_aia"]).astype(int)
mf=binary_metrics(y[fusion_ok],fusion_pred[fusion_ok])
comparators.append({
    "policy_id":"baseline_always_fusion",
    "requested_cases":len(y),
    "positives":int((y==1).sum()),
    "normal_cases":int(fusion_ok.sum()),
    "degraded_cases":0,
    "abstain_cases":int((~fusion_ok).sum()),
    "availability_rate":float(fusion_ok.mean()),
    "positive_retention_rate":float(((y==1)&fusion_ok).sum()/(y==1).sum()),
    "positive_abstention_rate":float(((y==1)&(~fusion_ok)).sum()/(y==1).sum()),
    "detected_flares":int(((y==1)&fusion_ok&(fusion_pred==1)).sum()),
    "issued_but_missed_flares":int(((y==1)&fusion_ok&(fusion_pred==0)).sum()),
    "abstained_flares":int(((y==1)&(~fusion_ok)).sum()),
    "false_alarms":int(((y==0)&fusion_ok&(fusion_pred==1)).sum()),
    **{f"issued_{k}":v for k,v in mf.items()}
})

sharp_pred=(sharp_p>=alarm_map["sharp"]).astype(int)
ms=binary_metrics(y[sharp_ok],sharp_pred[sharp_ok])
comparators.append({
    "policy_id":"baseline_always_sharp",
    "requested_cases":len(y),
    "positives":int((y==1).sum()),
    "normal_cases":0,
    "degraded_cases":int(sharp_ok.sum()),
    "abstain_cases":int((~sharp_ok).sum()),
    "availability_rate":float(sharp_ok.mean()),
    "positive_retention_rate":float(((y==1)&sharp_ok).sum()/(y==1).sum()),
    "positive_abstention_rate":float(((y==1)&(~sharp_ok)).sum()/(y==1).sum()),
    "detected_flares":int(((y==1)&sharp_ok&(sharp_pred==1)).sum()),
    "issued_but_missed_flares":int(((y==1)&sharp_ok&(sharp_pred==0)).sum()),
    "abstained_flares":int(((y==1)&(~sharp_ok)).sum()),
    "false_alarms":int(((y==0)&sharp_ok&(sharp_pred==1)).sum()),
    **{f"issued_{k}":v for k,v in ms.items()}
})

def eval_v3(pid,pct):
    fs=cutoff("fusion_seed_std",pct)
    gap=cutoff("sharp_aia_probability_gap",pct)
    ent=cutoff("fusion_entropy",pct)
    fusion_single=np.isin(
        sig["sharp_aia_mondrian_state"].to_numpy(),
        ["singleton_no_flare","singleton_flare"]
    )
    normal=(
        fusion_ok &
        fusion_single &
        (sig["fusion_seed_std"].to_numpy()<=fs) &
        (sig["sharp_aia_probability_gap"].to_numpy()<=gap) &
        (sig["fusion_entropy"].to_numpy()<=ent)
    )
    degraded=(~normal)&sharp_ok
    abstain=~(normal|degraded)
    require(np.all(normal.astype(int)+degraded.astype(int)+abstain.astype(int)==1),
            f"Non-exclusive v3 states {pid}")

    pred=np.full(len(y),-1,dtype=int)
    pred[normal]=(fusion_p[normal]>=alarm_map["sharp_aia"]).astype(int)
    pred[degraded]=(sharp_p[degraded]>=alarm_map["sharp"]).astype(int)
    issued=normal|degraded
    m=binary_metrics(y[issued],pred[issued])

    sig[f"{pid}_state"]=np.where(normal,"NORMAL",np.where(degraded,"DEGRADED","ABSTAIN"))
    sig[f"{pid}_prediction"]=pred

    positives=int((y==1).sum())
    return {
        "policy_id":pid,
        "cutoff_percentile":pct,
        "requested_cases":len(y),
        "positives":positives,
        "normal_cases":int(normal.sum()),
        "degraded_cases":int(degraded.sum()),
        "abstain_cases":int(abstain.sum()),
        "availability_rate":float(issued.mean()),
        "positive_retention_rate":float(((y==1)&issued).sum()/positives),
        "positive_abstention_rate":float(((y==1)&abstain).sum()/positives),
        "detected_flares":int(((y==1)&issued&(pred==1)).sum()),
        "issued_but_missed_flares":int(((y==1)&issued&(pred==0)).sum()),
        "abstained_flares":int(((y==1)&abstain).sum()),
        "false_alarms":int(((y==0)&issued&(pred==1)).sum()),
        **{f"issued_{k}":v for k,v in m.items()}
    }

rows=comparators.copy()
rows.append(eval_v3("fallback_v3_q80",0.80))
rows.append(eval_v3("fallback_v3_q90",0.90))
rows.append(eval_v3("fallback_v3_q95",0.95))
results=pd.DataFrame(rows)

primary=results.loc[results["policy_id"]=="fallback_v3_q90"].iloc[0]
fusion_base=results.loc[results["policy_id"]=="baseline_always_fusion"].iloc[0]
sharp_base=results.loc[results["policy_id"]=="baseline_always_sharp"].iloc[0]

comparison={
    "primary_policy_id":"fallback_v3_q90",
    "vs_always_fusion":{
        "availability_delta":float(primary["availability_rate"]-fusion_base["availability_rate"]),
        "positive_retention_delta":float(primary["positive_retention_rate"]-fusion_base["positive_retention_rate"]),
        "issued_tss_delta":float(primary["issued_tss"]-fusion_base["issued_tss"]),
        "false_alarm_delta":int(primary["false_alarms"]-fusion_base["false_alarms"]),
        "abstained_flare_delta":int(primary["abstained_flares"]-fusion_base["abstained_flares"]),
    },
    "vs_always_sharp":{
        "availability_delta":float(primary["availability_rate"]-sharp_base["availability_rate"]),
        "positive_retention_delta":float(primary["positive_retention_rate"]-sharp_base["positive_retention_rate"]),
        "issued_tss_delta":float(primary["issued_tss"]-sharp_base["issued_tss"]),
        "false_alarm_delta":int(primary["false_alarms"]-sharp_base["false_alarms"]),
        "abstained_flare_delta":int(primary["abstained_flares"]-sharp_base["abstained_flares"]),
    }
}

results_path=OUT/"supplementary2026_policy_results.csv"
coverage_path=OUT/"supplementary2026_mondrian_coverage.csv"
branch_path=OUT/"supplementary2026_branch_metrics.csv"
cases_path=OUT/"supplementary2026_policy_cases.csv.gz"
summary_path=OUT/"summary.json"

results.to_csv(results_path,index=False)
coverage.to_csv(coverage_path,index=False)
branch_metrics.to_csv(branch_path,index=False)
sig.to_csv(cases_path,index=False,compression={"method":"gzip","mtime":0})

summary={
    "status":"SUPPLEMENTARY_2026_FALLBACK_V3_CLEAN_EVALUATION_COMPLETE",
    "scientific_acceptance":False,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "role":ROLE,
    "rows":len(y),
    "positives":int((y==1).sum()),
    "primary_policy_id":"fallback_v3_q90",
    "policy_reselection_performed":False,
    "threshold_tuning_performed":False,
    "this_role_was_reserved_as_clean_future_evaluation":True,
    "comparison_primary":comparison,
    "source_sha256":{
        "master_predictions":MASTER_SHA,
        "sharp_predictions":SHARP_SHA,
        "mondrian_protocol":sha256(MOND_PROTOCOL),
        "mondrian_thresholds":sha256(MOND_THRESHOLDS),
        "candidate_cutoffs":sha256(CUTS),
        "alarm_protocol":sha256(ALARM_PROTOCOL),
        "alarm_thresholds":sha256(ALARM_THRESHOLDS),
        "v3_policy_protocol":sha256(V3_PROTOCOL),
        "v3_policy_family":sha256(V3_FAMILY),
    },
    "output_sha256":{
        results_path.name:sha256(results_path),
        coverage_path.name:sha256(coverage_path),
        branch_path.name:sha256(branch_path),
        cases_path.name:sha256(cases_path),
    }
}
summary_path.write_text(json.dumps(summary,indent=2)+"\n")

print("===== SUPPLEMENTARY 2026 FALLBACK-FIRST V3 CLEAN EVALUATION COMPLETE =====")
print("rows:",len(y))
print("positives:",int((y==1).sum()))
print("primary policy: fallback_v3_q90")
print("policy reselection performed: False")
print("threshold tuning performed: False")
print("reserved clean-future role: True")

print("\nBranch metrics:")
print(branch_metrics.to_string(index=False))

print("\nMondrian class-conditional coverage:")
print(coverage.to_string(index=False))

print("\nOperational policy results:")
print(results.to_string(index=False))

print("\nPrimary q90 comparisons:")
print(json.dumps(comparison,indent=2))

print("\nSaved:",summary_path)
print("Saved:",results_path)
print("Saved:",coverage_path)
print("Saved:",branch_path)
print("Saved:",cases_path)


===== SUPPLEMENTARY 2026 FALLBACK-FIRST V3 CLEAN EVALUATION COMPLETE =====
rows: 11118
positives: 992
primary policy: fallback_v3_q90
policy reselection performed: False
threshold tuning performed: False
reserved clean-future role: True

Branch metrics:
   branch  available_cases  availability_rate    brier  log_loss  tp   fp  fn   tn   recall  specificity  precision       f1      tss      hss
    sharp            11118                1.0 0.069099  0.281478 421 1028 571 9098 0.424395     0.898479   0.290545 0.344941 0.322874 0.267329
      aia            11118                1.0 0.086872  0.419158 116  438 876 9688 0.116935     0.956745   0.209386 0.150065 0.073680 0.092002
sharp_aia            11118                1.0 0.074304  0.303239 337  481 655 9645 0.339718     0.952499   0.411980 0.372376 0.292216 0.317319

Mondrian class-conditional coverage:
   branch  alpha  available_cases  overall_coverage  class0_coverage  class1_coverage  empty_rate  singleton_no_flare_rate  singleton_fl